# Lektion 05: 🧱 BUILD 1 – Der Wissensassistent in reinem Python



| | |
|---|---|
| **Zeit** | ~75 Minuten |
| **Schwierigkeit** | Mittel |
| **Voraussetzungen** | Lektion 01–04 |
| **Du baust** | Die erste Version deines Personal Knowledge Assistant – komplett von Hand, ohne jedes KI-Framework |

## 🎯 Nach dieser Lektion kannst du:

- Dokumente laden und in Abschnitte zerlegen

- eine eigene Suchfunktion (Wort-Überlappung) schreiben

- den Prompt **von Hand** bauen und das Modell damit füttern

- die manuelle Orchestrierung komplett erklären können

- erleben, wo diese Architektur an ihre Grenzen stößt

## ⚙️ Setup – einmal pro Notebook ausführen

Diese Zelle findet die Projektwurzel und lädt die **zentrale KI-Konfiguration** aus deiner `.env` (siehe `docs/local-ai-server.md`). Du musst nie Notebook-Code ändern, um den Anbieter zu wechseln.

In [1]:
import sys
from pathlib import Path

# Projektwurzel finden (dort liegt der Ordner "lernakademie")
ROOT = Path.cwd()
while not (ROOT / "lernakademie").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "lernakademie").exists():
    raise RuntimeError("Projektwurzel nicht gefunden – starte das Notebook im Repo-Ordner.")
sys.path.insert(0, str(ROOT))

from lernakademie.config import llm_zusammenfassung

_info = llm_zusammenfassung()  # zeigt NIE den API-Key selbst
print("KI-Konfiguration:", _info["anbieter"], "| Modell:", _info["modell"],
      "| Key:", _info["api_key"])
if _info["anbieter"] == "offline":
    print("Offline-Modus: alles laeuft – Antworten sind Simulationen ([OFFLINE]).")


KI-Konfiguration: offline | Modell: offline-simulation | Key: nicht gesetzt (optional)
Offline-Modus: alles laeuft – Antworten sind Simulationen ([OFFLINE]).


## 🧱 Willkommen zu BUILD 1!

Das ist der erste Meilenstein der Akademie. Du baust jetzt den **Personal Knowledge Assistant** zum **ersten** Mal – und zwar bewusst **komplett von Hand**:

- **KEIN** LangChain
- **KEIN** Framework
- Nur: Python, dein Wissen aus Lektion 01–04, und der direkte Modell-Aufruf

**WARUM so anstrengend?** Weil du erst **verstehen** sollst, was unter der Haube passiert, bevor dir ein Framework die Arbeit abnimmt. **Metapher:** Bevor du ein Fertighaus kaufst, baust du einmal einen Schuppen aus Holz. Danach weißt du, was LangChain (Build 2) wirklich für dich tut – und was es nur **versteckt**.

**Die Build-1-Architektur:**

```text
Dokumente (data/*.md)
   ↓  laden + in Abschnitte schneiden
Frage
   ↓  eigene Suche: Wort-Uebereinstimmung
beste Abschnitte
   ↓  Prompt von Hand zusammenbauen
LLM-Aufruf
   ↓
Antwort + Quellen
```

## Schritt 1: Dokumente laden

Im Ordner `data/` liegen drei erfundene "Firmen-Handbücher". Wir zerlegen jeden Text an seinen `## `-Überschriften in **Abschnitte** – jedes Mal ein Dictionary mit Quelle, Titel und Text (Lektion 01: Dictionarys!).

In [2]:
from pathlib import Path

DATA = ROOT / "data"

def lade_dokumente():
    """Liest alle data/*.md und schneidet sie an ## -Ueberschriften."""
    abschnitte = []
    for datei in sorted(DATA.glob("*.md")):
        text = datei.read_text(encoding="utf-8")
        teile = text.split("## ")
        for teil in teile[1:]:                # teile[0] ist der Dokumentkopf
            titel, *rest = teil.split("\n", 1)
            inhalt = rest[0].strip() if rest else ""
            if inhalt:
                abschnitte.append({
                    "quelle": datei.name,
                    "titel": titel.strip(),
                    "text": inhalt,
                })
    return abschnitte

abschnitte = lade_dokumente()
print(f"{len(abschnitte)} Abschnitte aus {len({a['quelle'] for a in abschnitte})} Dokumenten")
print("Beispiel:", abschnitte[0]["quelle"], "-", abschnitte[0]["titel"])

13 Abschnitte aus 3 Dokumenten
Beispiel: firmenprofil.md - Ueberblick


## Schritt 2: Die eigene Suche

**WARUM:** Der Assistent soll NUR aus den Dokumenten antworten. Dazu müssen wir die **relevantesten** Abschnitte finden – ohne irgendeine KI, nur mit Python.

**WAS (unser simples Verfahren):** Wort-Überlappung. Wir zählen, wie viele Wörter Frage und Abschnitt gemeinsam haben. **Metapher:** Wir vergleichen Wort-Zettel – wenn genug gleiche Zettel in Frage und Text stecken, passt der Abschnitt wahrscheinlich.

**TRY:** Schau dir das Scoring einmal roh an:

In [3]:
def _woerter(text):
    """Saeubert: klein, Umlaute normalisieren, Fuellwoerter (<4 Zeichen) raus."""
    text = text.lower().replace("ü", "ue").replace("ä", "ae").replace("ö", "oe")
    return {w.strip(".,?!:;\"'()") for w in text.split() if len(w) >= 4}

frage = "Wie viele Urlaubstage gibt es?"
print("Fragewörter:", sorted(_woerter(frage)))
print()
for abschnitt in abschnitte[:3]:
    treffer = _woerter(frage) & _woerter(abschnitt["text"])
    print(f"{abschnitt['titel']:20s} | Treffer: {sorted(treffer)}")

Fragewörter: ['gibt', 'urlaubstage', 'viele']

Ueberblick           | Treffer: []
Abteilung Engineering | Treffer: []
Abteilung Data & Analytics | Treffer: []


### 🔎 Was sehen wir hier?
Jede Frage wird zur **Wortmenge** (ein `set` – Mengen kennen keine Duplikate). Der Schnitt (`&`) zweier Mengen zeigt die gemeinsamen Wörter. Das ist der komplette "Geheimcode" unserer Build-1-Suche – mehr steckt nicht dahinter.

## ✏️ Deine Aufgabe: Die Funktion `suche_relevant`

Du schreibst das Herz der Build-1-Suche: eine Funktion, die zu einer Frage die besten Abschnitte liefert.

**Anforderungen:**

- Schreibe `suche_relevant(frage, abschnitte, top=2)`.

- Score = Anzahl gemeinsamer (gesäuberter) Wörter zwischen Frage und Abschnittstext.

- Nur Abschnitte mit Score > 0 kommen in die Ergebnisliste.

- Die Liste ist nach Score absteigend sortiert und maximal `top` lang.

**Schreibe deinen Code in die folgende Zelle:**

In [4]:
def suche_relevant(frage, abschnitte, top=2):
    frage_woerter = _woerter(frage)
    scored = []
    for abschnitt in abschnitte:
        score = len(frage_woerter & _woerter(abschnitt["text"]))
        if score > 0:
            scored.append((score, abschnitt))
    scored.sort(key=lambda paar: paar[0], reverse=True)
    return [abschnitt for _, abschnitt in scored[:top]]

for a in suche_relevant("Wie viele Urlaubstage gibt es?", abschnitte):
    print(a["quelle"], "-", a["titel"])

mitarbeiter-handbuch.md - Urlaub


**💡 Warum diese Loesung funktioniert:**

Jeder Abschnitt bekommt einen Punktwert (Score). Nach dem Sortieren schneiden wir die `top`-Besten ab. Die Funktion ist **deterministisch** – gleiche Frage, gleiche Treffer – deshalb konnte die Prüfung sie auch exakt testen. Determinismus ist Gold wert (Lektion 20 zeigt dir, wie man das systematisch nutzt).

### ✅ Richtig, wenn:

- [ ] Für die Urlaubsfrage liefert die Funktion den Abschnitt `mitarbeiter-handbuch.md – Urlaub` an erster Stelle.

- [ ] Fragen ohne Treffer ergeben eine **leere Liste**.

- [ ] Die Ausgabe ist eine Liste von Dictionaries (kein Score-Verlust in die Antwort!).

**🧪 Fuehre die folgende Zelle aus, um dein Ergebnis zu pruefen:**

In [5]:
treffer = suche_relevant("Wie viele Urlaubstage gibt es?", abschnitte)
assert treffer, "Die Urlaubsfrage sollte Treffer liefern!"
assert treffer[0]["titel"] == "Urlaub", f"Erster Treffer sollte 'Urlaub' sein, ist '{treffer[0]['titel']}'"
assert suche_relevant("Weihnachtsgeld Hoehe Bonus?", abschnitte) == [], "Ohne Treffer: leere Liste!"
assert len(suche_relevant("Urlaub Gehalt Passwort VPN", abschnitte, top=2)) <= 2
print("Alle Prüfungen bestanden! ✅")

Alle Prüfungen bestanden! ✅


<details><summary>💡 Hinweis 1 (oeffnen, wenn du nicht weiterkommst)</summary>

Baue eine Liste `(score, abschnitt)`, sortiere sie mit `sortiert = sorted(liste, key=lambda paar: paar[0], reverse=True)`.
</details>

<details><summary>💡 Hinweis 2 (oeffnen, wenn du nicht weiterkommst)</summary>

Die Bereinigung erledigt die Hilfsfunktion `_woerter` aus der Zelle oben (du darfst sie benutzen!).
</details>

<details><summary>💡 Hinweis 3 (oeffnen, wenn du nicht weiterkommst)</summary>

Vergiss nicht: nur Score > 0 mitnehmen – sonst kämen bei jeder Frage alle Abschnitte durch.
</details>

## 📝 Reflexion

Beantworte in deinen eigenen Worten:

1. Deine Suche findet "Urlaub" – aber wie gut findet sie **"freie Tage"**? Wo liegt die Schwäche (und was wird Build 2 dagegen tun)?

2. Was passiert bei einer Frage voller kleiner Wörter ("Wie ist das mit dem?") – und warum haben wir Wörter < 4 Zeichen aussortiert?

## Schritt 3: Der Prompt von Hand

**WARUM:** Das Modell soll **nur aus den gefundenen Abschnitten** antworten – sonst halluziniert es. Deshalb bauen wir den Kontext **mit Anweisung** in einen Text.

**Genau diese String-Arbeit** wird in Build 2 `ChatPromptTemplate` übernehmen – merk dir, wie sie von Hand aussieht:

In [6]:
def baue_prompt(frage, abschnitte_liste):
    kontext = "\n\n".join(
        f"[{a['quelle']} - {a['titel']}]\n{a['text']}" for a in abschnitte_liste)
    return (
        "Du bist ein Assistent fuer Firmenhandbuecher. Antworte NUR auf Basis "
        "des folgenden Kontexts. Wenn die Antwort nicht im Kontext steht, sage: "
        "'Das steht in den Dokumenten leider nicht.'\n\n"
        f"Kontext:\n{kontext}\n\nFrage: {frage}"
    )

beispiel = baue_prompt("Urlaub?", abschnitte[:2])
print(beispiel[:300] + "...")

Du bist ein Assistent fuer Firmenhandbuecher. Antworte NUR auf Basis des folgenden Kontexts. Wenn die Antwort nicht im Kontext steht, sage: 'Das steht in den Dokumenten leider nicht.'

Kontext:
[firmenprofil.md - Ueberblick]
Die Beispielfirma GmbH ist ein Technologieunternehmen mit Sitz in
Berlin. S...


## Schritt 4: Alles orchestrieren

Jetzt der **Dirigent**: Suche → Prompt → Modell → Antwort + Quellen. Das ist die komplette manuelle Orchestrierung deines Assistenten:

In [7]:
from lernakademie.llm_client import get_api_client, get_model_name

NICHT_GEFUNDEN = "Dazu habe ich in den Dokumenten leider nichts gefunden."

def beantworte(frage, abschnitte):
    relevante = suche_relevant(frage, abschnitte)
    if not relevante:
        return {"antwort": NICHT_GEFUNDEN, "quellen": []}
    prompt = baue_prompt(frage, relevante)
    client = get_api_client()
    try:
        antwort = client.chat.complete(
            model=get_model_name(),
            messages=[{"role": "user", "content": prompt}])
        text = antwort.choices[0].message.content
    except Exception as fehler:
        return {"antwort": f"Technisches Problem: {fehler}", "quellen": []}
    quellen = [f"{a['quelle']} - {a['titel']}" for a in relevante]
    return {"antwort": text, "quellen": quellen}

for frage in ("Wie viele Urlaubstage gibt es?",
              "Was gilt fuer Passwoerter?",
              "Wie hoch ist das Weihnachtsgeld?"):
    ergebnis = beantworte(frage, abschnitte)
    print("FRAGE  :", frage)
    print("ANTWORT:", ergebnis["antwort"][:120])
    print("QUELLEN:", ergebnis["quellen"], "\n")

FRAGE  : Wie viele Urlaubstage gibt es?
ANTWORT: [OFFLINE] Simulierte Antwort (Offline-Modus) auf: „Du bist ein Assistent fuer Firmenhandbuecher. Antworte NUR auf Basis 
QUELLEN: ['mitarbeiter-handbuch.md - Urlaub'] 

FRAGE  : Was gilt fuer Passwoerter?
ANTWORT: [OFFLINE] Simulierte Antwort (Offline-Modus) auf: „Du bist ein Assistent fuer Firmenhandbuecher. Antworte NUR auf Basis 
QUELLEN: ['mitarbeiter-handbuch.md - Kündigung', 'firmenprofil.md - Ueberblick'] 

FRAGE  : Wie hoch ist das Weihnachtsgeld?
ANTWORT: Dazu habe ich in den Dokumenten leider nichts gefunden.
QUELLEN: [] 



### 🔎 Was sehen wir hier?
1. Die Urlaubsfrage findet den richtigen Abschnitt (Quelle: `mitarbeiter-handbuch.md – Urlaub`).
2. Die Passwort-Frage findet den IT-Abschnitt.
3. Die Weihnachtsgeld-Frage ist ein **Beweis für Ehrlichkeit**: kein Treffer → keine erfundene Antwort.

**Dieselbe Version läuft jetzt als wiederverwendbares Modul** (`lernakademie/build1.py`) – genau das ist der "Level 3"-Schritt aus dem Notebook-Guide: Konzept erst von Hand verstehen, dann als Abstraktion nutzen. Importieren wir sie:

In [8]:
from lernakademie.build1 import beantworte as beantworte_modul, lade_dokumente

ergebnis = beantworte_modul("Wie viele Urlaubstage gibt es?", lade_dokumente())
print("Antwort:", ergebnis["antwort"][:150])
print("Quellen:", ergebnis["quellen"])

Antwort: [OFFLINE] Simulierte Antwort (Offline-Modus) auf: „Du bist ein Assistent fuer Firmenhandbuecher. Antworte NUR auf Basis des folgenden Kontexts. Wenn d
Quellen: ['mitarbeiter-handbuch.md - Urlaub']


## 🧩 Challenge (freiwillig): Die Grenze erleben

Stelle die Frage **"Wie beantrage ich freie Tage?"** – es steht kein Wort davon wörtlich in den Dokumenten. Beobachte, was deine Wort-Suche tut. Schreib deine Beobachtung als Kommentar in die Zelle. Genau diese Schwäche wird Build 2 mit **Embeddings** (Bedeutungs-Suche) beheben!

In [9]:
ergebnis = beantworte("Wie beantrage ich freie Tage?", abschnitte)
print("ANTWORT:", ergebnis["antwort"][:150])
print("QUELLEN:", ergebnis["quellen"])
# DEINE BEOBACHTUNG: ...


ANTWORT: Dazu habe ich in den Dokumenten leider nichts gefunden.
QUELLEN: []


## ➡️ Weiter

**Weiter mit Lektion 06:** `notebooks/03_build_1/06_build1_erweiterung_ui.ipynb` – du erweiterst Build 1 um eine zweite Wissensquelle, dokumentierst den Wartungsschmerz und startest deine erste Web-Oberfläche.